# Liquidity Management Calibrations
1. **Liquidity profile** — days-to-liquidate, bucket distribution, redemption stress, investor concentration, liquidity-adjusted VaR
2. **LMT trigger analysis** — 12-month gate / swing / suspension simulation per AIFMD II Art. 16a

In [ ]:
from src.data.database   import get_engine
from src.risk.risk_utils import redemption_stress

ENGINE  = get_engine()


In [ ]:
# in teis repo asll computation refer to the same date
from src.config import VALUATION_DATE
VALUATION_DATE

### 1. Liquidity Profile

Days-to-liquidate and ESMA bucket distribution across all six funds. Positions loaded from the database at the standard valuation date.


In [ ]:
from src.config import VALUATION_DATE, LIQUIDITY_BUCKET_ORDER

OPEN_FUNDS = {
    'UCITS_Balanced':   'UCITS Balanced',
    'AIFM_HedgeFund':   'AIFM Hedge Fund',
}

# Load positions and create liquidity bucket data
from src.data.database import query_positions
from src.risk.risk_utils import days_to_liquidate, liquidity_buckets
from src.ui.liquidity_calibration_display import plot_liquidity_profile_chart

funds_data = {}
for fund_id in OPEN_FUNDS.keys():
    pos = query_positions(ENGINE, fund_id, VALUATION_DATE)
    pos_liq = liquidity_buckets(days_to_liquidate(pos, pct_adv=0.25))
    funds_data[OPEN_FUNDS[fund_id]] = pos_liq

plot_liquidity_profile_chart(funds_data, VALUATION_DATE, LIQUIDITY_BUCKET_ORDER, export_id="01")

#### 1.2 Redemption Stress — Single-Period Snapshot

Point-in-time gate analysis: can each open-ended fund meet a 25% redemption within a 5-day notice period? PE Buyout and Infra Core are excluded — no redemption obligation.


In [ ]:
from src.ui.liquidity_calibration_display import display_redemption_stress_table

stress_results = {}
for fund_id, fund_name in OPEN_FUNDS.items():
    pos = query_positions(ENGINE, fund_id, VALUATION_DATE)
    nav = pos['market_value_eur'].sum()
    pos_liq = liquidity_buckets(days_to_liquidate(pos, pct_adv=0.25))
    rs = redemption_stress(pos_liq, nav, redemption_pct=0.25, notice_days=5)
    stress_results[fund_name] = {
        'nav': nav,
        'redemption_amount_eur': rs['redemption_amount_eur'],
        'liquid_assets_eur': rs['liquid_assets_eur'],
        'coverage_ratio': rs['coverage_ratio'],
        'can_meet_redemption': rs['can_meet_redemption'],
    }

display_redemption_stress_table(stress_results, VALUATION_DATE, export_id="02")

### 2 Investor Base Model -- Modeling the Redemption Schedule

Rather than specifying a redemption schedule as a fixed list of numbers, we derive it from first principles using a per-fund **investor registry**. Each fund's investor base is decomposed into investor types, each characterised by:

| Notation | Meaning |
|---|---|
| $w_i$ | AUM share of investor type $i$ ($\sum_i w_i = 1$) |
| $\mu_i^{\text{norm}}$ | Monthly redemption rate under normal conditions |
| $\mu_i^{\text{stress}}$ | Monthly redemption rate under stress conditions |
| $\sigma$ | Lognormal dispersion parameter (per fund) |

#### Normal months -- lognormal draws

For months not designated as stress months, each investor type draws from a lognormal distribution centred on its normal rate. The lognormal is chosen because redemption rates are strictly positive and right-skewed -- occasional large outflows occur with low but non-negligible probability:

$$X_{i,t} \sim \text{LogNormal}\!\left(\ln\mu_i^{\text{norm}} - \tfrac{\sigma^2}{2},\;\; \sigma\right)$$

Parametrised this way, $\mathbb{E}[X_{i,t}] = \mu_i^{\text{norm}}$ exactly. The dispersion $\sigma$ is calibrated higher for hedge fund investor bases, which exhibit more volatile redemption behaviour than retail or balanced fund investors.

The aggregate monthly redemption rate is the AUM-weighted sum across all investor types:

$$r_t = \sum_{i=1}^{N} w_i \cdot X_{i,t}$$

#### Stress months -- deterministic override

Designated stress months $t \in \mathcal{T}_{\text{stress}}$ override the lognormal draw and use each investor type's stress rate directly:

$$r_t = \sum_{i=1}^{N} w_i \cdot \mu_i^{\text{stress}} \quad \forall\; t \in \mathcal{T}_{\text{stress}}$$

This models a coordinated redemption wave -- e.g. following a market shock, prime broker margin call, or manager headline risk -- where all investor types simultaneously redeem at elevated rates. Stress months are parameterised per fund.

#### Weighted reference rates

The two reference lines on the schedule chart are:

$$r^{\text{norm,wt}} = \sum_i w_i \cdot \mu_i^{\text{norm}} \qquad r^{\text{stress,wt}} = \sum_i w_i \cdot \mu_i^{\text{stress}}$$

These are the expected aggregate rates under normal and full-stress conditions respectively.

In [ ]:
# Load investor registers and calibration inputs
from src.data.reference_data import (
    load_investor_base_dict,
    load_liquidity_calibration_inputs,
)
from src.computation.liquidity_calibration import (
    build_redemption_schedule,
    compute_redemption_scenarios,
)

FUNDS = {
    'UCITS_Balanced': 'UCITS Balanced',
    'AIFM_HedgeFund': 'AIFM Hedge Fund',
}

# Load investor bases and calibration inputs
investor_inputs = {}
calibration_inputs = {}
redemption_scenarios = {}
largest_investor_scenario = {}

for fund_id, fund_label in FUNDS.items():
    investor_inputs[fund_id] = load_investor_base_dict(fund_id)
    raw_calib = load_liquidity_calibration_inputs(fund_id)
    calibration_inputs[fund_id] = raw_calib
    
    # Normalize calibration config structure
    # UCITS has investors at top level; HF has it nested in redemption_schedule_calibration
    if 'redemption_schedule_calibration' in raw_calib:
        calib_config = raw_calib['redemption_schedule_calibration']
    else:
        # UCITS structure: investors, stress_months, sigma, seed at top level
        calib_config = {
            'investors': raw_calib.get('investors', []),
            'stress_months': raw_calib.get('stress_months', []),
            'sigma': raw_calib.get('sigma', 0.3),
            'seed': raw_calib.get('seed', 42),
        }
    
    # Compute redemption scenarios
    scenarios_result = compute_redemption_scenarios(
        investor_inputs[fund_id],
        calib_config
    )
    redemption_scenarios[fund_id] = scenarios_result['redemption_scenarios']
    largest_investor_scenario[fund_id] = {
        'name': 'Largest investor',
        'pct': scenarios_result['largest_investor_pct'],
        'investor_name': scenarios_result['largest_investor_name'],
    }

print("✓ Loaded investor registers and computed redemption scenarios for both funds")

### 3. LMT Trigger Analysis
**AIFMD II (MRS-84)**

The simulation applies to **open-ended funds with a liquid portfolio only**: UCITS Balanced and AIFM Hedge Fund.  
AIFM Private Debt and AIFM Real Estate hold predominantly illiquid assets (loans and direct property). For those funds any redemption wave rapidly exhausts the liquid buffer in the first one or two gated periods; the analytically interesting question there is *how many months until the liquid sleeve hits zero*, not the gate/swing dynamics — see Section 3.  
PE Buyout and Infra Core are closed-ended: no redemption obligation, no LMT tools required.

---

#### Regulatory basis

- **AIFMD II** — Directive 2024/927/EU, Art. 16a: open-ended AIFs must activate at least two LMTs in stress
- **ESMA Guidelines** — ESMA34-671404336-1364 (April 2025): defines activation conditions, board governance requirements, and reporting obligations
- **Del. Reg. 231/2013** — Articles 46–49: liquidity management framework; Art. 95: LMT policy content

---

#### Three LMTs modelled

| Tool | Type | Trigger condition | Action |
|---|---|---|---|
| Redemption gate | Quantitative | Gross demand $> G \cdot V_t$ | Cap paid at $\min(G \cdot V_t,\; L_t)$; defer excess |
| Swing pricing | Anti-dilution | Gross rate $> S$ | Adjust NAV per unit by levy $\delta$ |
| Suspension | Exceptional | Board decision (simulated: two conditions) | Zero redemptions paid |

where $G$ = gate threshold, $V_t$ = total NAV at $t$, $L_t$ = liquid sleeve at $t$, $S$ = swing threshold, $\delta$ = swing factor.

---

#### Model mechanics

##### 1. Contagion feedback

When the gate fires in period $t-1$, rational remaining investors accelerate their own redemption requests to avoid being queue-prioritised behind a growing backlog. This is modelled by scaling the base schedule:

$$r_t^{\text{eff}} = \begin{cases} \min\!\left(r_t^{\text{base}} \cdot \lambda,\; 1\right) & \text{if gate was active in } t-1 \\ r_t^{\text{base}} & \text{otherwise} \end{cases}$$

where $\lambda \geq 1$ is the **contagion multiplier**. Calibration: $\lambda = 1.3$ for UCITS (mixed retail/institutional); $\lambda = 1.5$ for hedge funds (concentrated institutional base with faster reaction time).

The effective gross demand in EUR is then:

$$D_t^{\text{new}} = r_t^{\text{eff}} \cdot V_t$$

##### 2. Gate mechanics

Total demand includes new requests and the outstanding backlog $B_{t-1}$:

$$D_t^{\text{total}} = D_t^{\text{new}} + B_{t-1}$$

The **gate cap** is the lower of the contractual limit (a fraction of total NAV) and what the fund can actually liquidate:

$$C_t = \min\!\left(G \cdot V_t,\; L_t\right)$$

The gate fires when $D_t^{\text{total}} > C_t$. When active:

$$\text{Paid}_t = \min(C_t,\; D_t^{\text{total}})$$

$$\text{Deferred}_t = \max\!\left(0,\; D_t^{\text{new}} - \max(0,\; C_t - B_{t-1})\right)$$

$$B_t = \max\!\left(0,\; D_t^{\text{total}} - \text{Paid}_t\right)$$

The liquid sleeve evolves as:

$$L_t = L_{t-1} - \text{Paid}_t$$

##### 3. Swing pricing

The swing activates when $r_t^{\text{eff}} > S$. A dilution levy $\delta$ is applied to the effective NAV per unit, adjusting the redemption price upward so that transaction costs are borne by redeeming investors rather than the remaining fund:

$$\text{NAV}_t^{\text{eff}} = V_t \cdot (1 + \delta)$$

This does not reduce the fund's cash outflow — it increases the effective redemption price, protecting remaining investors from dilution.

##### 4. Suspension (simulation approximation)

Suspension triggers automatically in the model when two conditions hold simultaneously:

$$\text{consec\_gate} \geq N_{\text{suspend}} \quad \text{AND} \quad \frac{B_t}{L_t} \geq \theta_B$$

where $N_{\text{suspend}}$ is the minimum consecutive gate months and $\theta_B$ is the backlog-to-liquid-NAV threshold.

**Important**: under AIFMD II and ESMA Guidelines (ESMA34-671404336-1364), actual suspension requires exceptional circumstances and a formal board decision — it cannot be automatic. The auto-trigger is a simulation convenience for stress-testing only.

#### 3.1 UCITS Balanced

**Portfolio**: multi-asset (equities, IG/HY bonds, listed alternatives). Liquid sleeve estimated at **85% of NAV** — the remaining 15% is allocated to less-liquid instruments (HY bonds and listed alternatives with lower daily turnover).

**Parameters**:

| Parameter | Value | Rationale |
|---|---|---|
| Gate threshold $G$ | 10% of NAV | Standard UCITS gate; contractual fund document limit |
| Swing threshold $S$ | 5% of NAV | Activates when net flows are large enough to impose material transaction costs |
| Swing factor $\delta$ | 50 bps | Market impact estimate for the liquid sleeve |
| Contagion multiplier $\lambda$ | 1.3 | Mixed retail/institutional base; slower reaction than pure HF |
| Suspension conditions | $N = 3$ consecutive gate months; $\theta_B = 25\%$ | Conservative — UCITS retail investor protections favour later, deliberate board decision |

**Stress schedule**: two consecutive spike months (months 3–4 at 10–14% of NAV) followed by sustained moderate outflow. Total 12-month gross request ≈ 61% of initial NAV.

In [ ]:
# Load LMT calibration inputs and build redemption schedules
from src.risk.risk_utils import lmt_trigger_analysis

LMT_PARAMS = {}

for fund_id, fund_label in FUNDS.items():
    raw_calib = calibration_inputs[fund_id]
    
    # Normalize calibration config structure (same as cell 8)
    if 'redemption_schedule_calibration' in raw_calib:
        calib_config = raw_calib['redemption_schedule_calibration']
    else:
        # UCITS structure: investors, stress_months, sigma, seed at top level
        calib_config = {
            'investors': raw_calib.get('investors', []),
            'stress_months': raw_calib.get('stress_months', []),
            'sigma': raw_calib.get('sigma', 0.3),
            'seed': raw_calib.get('seed', 42),
        }
    
    # Build 12-month redemption schedule
    schedule = build_redemption_schedule(calib_config, n_months=12)
    
    # Extract LMT thresholds (if present in lmt_calibration section)
    lmt_config = raw_calib.get('lmt_calibration', {})
    
    # Extract contractual and stress terms
    contractual = raw_calib.get('contractual_terms', {})
    stress_assumptions = raw_calib.get('stress_assumptions', {})
    
    LMT_PARAMS[fund_id] = {
        'label': fund_label,
        'schedule': schedule,
        'contractual_terms': contractual,
        'stress_assumptions': stress_assumptions,
        'lmt_thresholds': lmt_config,
    }

print("✓ Loaded LMT parameters and redemption schedules")

In [ ]:
# UCITS Balanced — LMT Trigger Analysis

from src.ui.liquidity_calibration_display import (
    display_redemption_scenarios,
    display_lmt_summary,
    plot_lmt_analysis,
)
from src.data.database import query_positions

# LMT parameters (from risk policy and liquidity governance)
UCITS_LMT_CONFIG = {
    'liquid_pct': 0.85,
    'gate_threshold': 0.10,
    'swing_threshold': 0.05,
    'contagion': 1.3,
    'consec_gate': 3,
    'backlog_pct': 0.25,
}

p = LMT_PARAMS['UCITS_Balanced']
pos = query_positions(ENGINE, 'UCITS_Balanced', VALUATION_DATE)
nav = pos['market_value_eur'].sum()

df_ucits = lmt_trigger_analysis(
    nav=nav,
    liquid_pct=UCITS_LMT_CONFIG['liquid_pct'],
    gate_threshold=UCITS_LMT_CONFIG['gate_threshold'],
    swing_threshold=UCITS_LMT_CONFIG['swing_threshold'],
    redemption_schedule=p['schedule'],
    consecutive_gate_for_suspension=UCITS_LMT_CONFIG['consec_gate'],
    backlog_pct_for_suspension=UCITS_LMT_CONFIG['backlog_pct'],
    contagion_multiplier=UCITS_LMT_CONFIG['contagion'],
)

# Display redemption scenarios
scenarios_data = {
    'redemption_scenarios': redemption_scenarios['UCITS_Balanced'],
    'largest_investor_name': largest_investor_scenario['UCITS_Balanced']['investor_name'],
    'largest_investor_pct': largest_investor_scenario['UCITS_Balanced']['pct'],
}

display_redemption_scenarios(scenarios_data, 'UCITS_Balanced', VALUATION_DATE)
display_lmt_summary(df_ucits, 'UCITS_Balanced', VALUATION_DATE)
plot_lmt_analysis(df_ucits, 'UCITS_Balanced', VALUATION_DATE)

#### 3.2 AIFM Hedge Fund

**Portfolio**: long/short equity with derivatives overlay. Liquid sleeve estimated at **75% of NAV** — the remaining 25% consists of less-liquid mid/small-cap positions, structured products, and OTC derivatives with longer unwind horizons.

**Parameters**:

| Parameter | Value | Rationale |
|---|---|---|
| Gate threshold $G$ | 10% of NAV | Standard AIFMD gate; fund document limit |
| Swing threshold $S$ | 5% of NAV | Same level as UCITS; fund is similarly liquid on the asset side |
| Swing factor $\delta$ | 50 bps | Market impact estimate |
| Contagion multiplier $\lambda$ | 1.5 | Concentrated institutional investor base (pension funds, family offices, funds-of-funds); faster and larger reaction when gate fires |
| Suspension conditions | $N = 3$ consecutive gate months; $\theta_B = 25\%$ | Same threshold as UCITS; board governance applies equally |

**Stress schedule**: slightly more severe than UCITS — month 4 peaks at 15% gross — reflecting the tendency of institutional HF investors to concentrate redemptions during market dislocations (liquidity crisis, manager performance concerns). Total 12-month gross request ≈ 70% of initial NAV before contagion scaling.

**Key difference from UCITS**: the higher $\lambda = 1.5$ means a gate in month $t$ can amplify the month $t+1$ request by 50%, making early gate activation a self-reinforcing dynamic. This is the primary reason HF liquidity stress is structurally more severe than UCITS despite a similar asset-side profile.

In [ ]:
# AIFM Hedge Fund — LMT Trigger Analysis

from src.data.database import query_positions

# LMT parameters (from risk policy and liquidity governance)
HF_LMT_CONFIG = {
    'liquid_pct': 0.75,
    'gate_threshold': 0.10,
    'swing_threshold': 0.05,
    'contagion': 1.5,
    'consec_gate': 3,
    'backlog_pct': 0.25,
}

p = LMT_PARAMS['AIFM_HedgeFund']
pos = query_positions(ENGINE, 'AIFM_HedgeFund', VALUATION_DATE)
nav = pos['market_value_eur'].sum()

df_hf = lmt_trigger_analysis(
    nav=nav,
    liquid_pct=HF_LMT_CONFIG['liquid_pct'],
    gate_threshold=HF_LMT_CONFIG['gate_threshold'],
    swing_threshold=HF_LMT_CONFIG['swing_threshold'],
    redemption_schedule=p['schedule'],
    consecutive_gate_for_suspension=HF_LMT_CONFIG['consec_gate'],
    backlog_pct_for_suspension=HF_LMT_CONFIG['backlog_pct'],
    contagion_multiplier=HF_LMT_CONFIG['contagion'],
)

# Display redemption scenarios
scenarios_data = {
    'redemption_scenarios': redemption_scenarios['AIFM_HedgeFund'],
    'largest_investor_name': largest_investor_scenario['AIFM_HedgeFund']['investor_name'],
    'largest_investor_pct': largest_investor_scenario['AIFM_HedgeFund']['pct'],
}

display_redemption_scenarios(scenarios_data, 'AIFM_HedgeFund', VALUATION_DATE)
display_lmt_summary(df_hf, 'AIFM_HedgeFund', VALUATION_DATE)
plot_lmt_analysis(df_hf, 'AIFM_HedgeFund', VALUATION_DATE)

### 4. Cross-Fund Summary


In [ ]:
from src.ui.liquidity_calibration_display import display_lmt_cross_fund_summary

lmt_results = {
    'UCITS Balanced': df_ucits,
    'AIFM Hedge Fund': df_hf,
}

display_lmt_cross_fund_summary(lmt_results, VALUATION_DATE, export_id="16")

## Suggested Liquidity Policy Blocks

The notebook suggests liquidity monitoring parameters that can be reviewed and manually added to each fund's risk_policy.json if appropriate.

In [ ]:
# Suggested UCITS liquidity monitoring block
from src.ui.liquidity_calibration_display import suggest_liquidity_policy_block, display_suggested_policy_block

ucits_policy = suggest_liquidity_policy_block(
    "UCITS_Balanced",
    {
        "redemption_scenarios": redemption_scenarios["UCITS_Balanced"],
        "largest_investor_name": largest_investor_scenario["UCITS_Balanced"]["investor_name"],
        "largest_investor_pct": largest_investor_scenario["UCITS_Balanced"]["pct"],
    },
    nav,
    notice_period_days=1,
    stress_window_days=5,
)

display_suggested_policy_block(ucits_policy, "UCITS_Balanced")


In [ ]:
# Suggested AIFM_HedgeFund liquidity monitoring block
hf_policy = suggest_liquidity_policy_block(
    "AIFM_HedgeFund",
    {
        "redemption_scenarios": redemption_scenarios["AIFM_HedgeFund"],
        "largest_investor_name": largest_investor_scenario["AIFM_HedgeFund"]["investor_name"],
        "largest_investor_pct": largest_investor_scenario["AIFM_HedgeFund"]["pct"],
    },
    nav,
    notice_period_days=30,
    stress_window_days=30,
)

display_suggested_policy_block(hf_policy, "AIFM_HedgeFund")
